## Real Estate Analysis (ZIP 90004)
Credentials come from .env 

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv("../.env")
engine = create_engine(f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
                       f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT','5432')}/{os.getenv('DB_NAME')}?sslmode=require")


df=pd.read_sql("select * from student_project.properties_cleaned",engine)
print(df.shape)
df.head()

# How do price vary by property type

In [ ]:
summary=(df.groupby("property_type")["list_price"]
         .agg(listing="count",average="mean",median="median")
         .round(0).sort_values("median",ascending=False))
summary

In [ ]:
ax=summary[["average","median"]].plot(kind="bar",figsize=(8,4))
ax.set_title("list price by property type (ZIP 90004)")
ax.set_ylabel("USD")
plt.tight_layout()
plt.show

FINDINGS:In ZIP 90004,single_family($3.17M average)and multi_family($2.60M average)properties command the highest average list prices,while condos are the most affordable at an $861k average ($759k median).Notably,single-family and multi-family averages are substantially higher than their medians (~$1.89M and $1.84M),indicating high-end luxury outliers pulling the mean upward,whereas townhomes reflect only a single listing ($1.05M).

## Price tiers

In [ ]:
bins=[0,750_000,1_500_000,3_500_000,5_000_000,float("inf")]
labels=["1.Entry","2.Mid_range","3.High_end","4.Luxury","5.Ultra-luxury"]
df["price_tier"]=pd.cut(df["list_price"],bins=bins,labels=labels,right=False)
tiers=df["price_tier"].value_counts().sort_index()
tiers.plot(kind="bar",figsize=(7,3),title="Listing price per tier")
plt.tight_layout();plt.show()
tiers

CONCLUSION: Most listings fall into the High-end (45 listings) and Mid-range (42 listings) tiers, while Luxury has the fewest (9 listings), followed closely by Entry (10 listings) and Ultra-luxury (13 listings). This shows that the market is concentrated in the middle-to-upper-middle price range rather than entry-level or luxury homes.

3.Most expensive listing

In [ ]:
df.sort_values("list_price",ascending=False).head(5)[["property_type","property_id","list_price","price_tier"]]

4.Outliers

In [ ]:
print(df["is_outlier"].sum(),"outliers listings")
df.groupby("is_outlier")["list_price"].agg(["count","mean","median"]).round(0)

Finding: The 25 outlier listings have a much higher average price ($5.01M) than the remaining 94 listings ($1.89M), which pulls the overall market average up. As a result, the median ($1.47M for non-outliers) gives a more accurate picture of typical home prices than the overall average.

In [ ]:
import os
os.makedirs("../powerbi",exist_ok=True)
df.to_csv("../powerbi/powerbi_data.csv",index=False)